# Embeddings estructurados mediante optimización restringida

Este notebook reformula el experimento para comparar **dos escenarios bajo la misma función objetivo y las mismas restricciones**:

1. Un caso sintético de notas musicales generado con una cadena de Markov.
2. Un caso clásico de NLP basado en un fragmento de *Don Quijote de la Mancha*.

La idea es que el caso sintético siga siendo útil como control, pero que el caso clásico produzca patrones de co-ocurrencia más interpretables. Así podemos comparar no solo los **métodos numéricos** (penalización cuadrática y barrera logarítmica), sino también el efecto de usar datos más ricos.


## 1. Importación de librerías

Usaremos `torch` para la optimización automática, `numpy` para simulación y `matplotlib` para visualización. Si el kernel no encuentra `torch`, la celda mostrará la ruta del Python activo para instalarlo en ese mismo entorno.


In [ ]:
import math
import random
import re
import sys
from collections import Counter
from textwrap import dedent

import matplotlib.pyplot as plt
import numpy as np

try:
    import torch
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    print("No se encontró 'torch' en este kernel de Jupyter.")
    print(f"Python activo: {sys.executable}")
    print("Instálalo en este mismo entorno con:")
    print(f"{sys.executable} -m pip install torch")
    raise exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)


## 2. Construcción de datasets comparables

El notebook ya no usa un único conjunto sintético. Ahora construimos dos secuencias discretas:

- `musica_sintetica`: conserva el ejemplo original y sirve como línea base controlada.
- `quijote_clasico`: usa un fragmento clásico de *Don Quijote* tokenizado y filtrado para dejar nombres y conceptos semánticos relevantes.

En ambos casos se construyen pares positivos `(centro, contexto)` con una ventana local. Eso permite mantener intacta la formulación tipo skip-gram y comparar resultados con justicia.


In [ ]:
def build_positive_pairs(sequence, window_size=2):
    # Construye pares positivos (centro, contexto) usando una ventana simétrica.
    pairs = []
    for t, center in enumerate(sequence):
        left = max(0, t - window_size)
        right = min(len(sequence), t + window_size + 1)
        for s in range(left, right):
            if s != t:
                pairs.append((center, sequence[s]))
    return pairs


def prepare_dataset_from_sequence(tokens, dataset_name, window_size=2, min_count=1, keep_tokens=None):
    counts = Counter(tokens)
    if keep_tokens is None:
        vocab = sorted([tok for tok, count in counts.items() if count >= min_count])
    else:
        vocab = [tok for tok in keep_tokens if counts[tok] >= min_count]

    token_to_id = {tok: i for i, tok in enumerate(vocab)}
    filtered_sequence = [token_to_id[tok] for tok in tokens if tok in token_to_id]
    pairs = build_positive_pairs(filtered_sequence, window_size=window_size)
    pair_counts = Counter(pairs)

    positive_pairs = torch.tensor(list(pair_counts.keys()), dtype=torch.long)
    positive_weights = torch.tensor(list(pair_counts.values()), dtype=torch.float32)
    positive_weights = positive_weights / positive_weights.mean()

    return {
        "name": dataset_name,
        "vocab": vocab,
        "token_to_id": token_to_id,
        "id_to_token": {i: tok for tok, i in token_to_id.items()},
        "sequence": filtered_sequence,
        "positive_pairs": positive_pairs,
        "positive_weights": positive_weights,
        "pair_counts": pair_counts,
        "n_tokens": len(vocab),
        "window_size": window_size,
    }


def build_music_case():
    vocab = ["C", "D", "E", "F", "G", "A", "B", "C_high"]
    P = np.array([
        [0.05, 0.25, 0.20, 0.05, 0.25, 0.05, 0.05, 0.10],
        [0.10, 0.05, 0.30, 0.10, 0.20, 0.10, 0.05, 0.10],
        [0.15, 0.10, 0.05, 0.25, 0.20, 0.10, 0.05, 0.10],
        [0.10, 0.05, 0.15, 0.05, 0.35, 0.10, 0.10, 0.10],
        [0.30, 0.05, 0.10, 0.10, 0.05, 0.20, 0.05, 0.15],
        [0.15, 0.10, 0.10, 0.05, 0.30, 0.05, 0.15, 0.10],
        [0.25, 0.05, 0.10, 0.05, 0.20, 0.10, 0.05, 0.20],
        [0.35, 0.10, 0.10, 0.05, 0.20, 0.05, 0.05, 0.10],
    ])
    P = P / P.sum(axis=1, keepdims=True)

    T = 2500
    sequence_ids = [0]
    for _ in range(T - 1):
        current = sequence_ids[-1]
        sequence_ids.append(np.random.choice(len(vocab), p=P[current]))

    tokens = [vocab[idx] for idx in sequence_ids]
    dataset = prepare_dataset_from_sequence(
        tokens,
        dataset_name="musica_sintetica",
        window_size=2,
        keep_tokens=vocab,
    )
    dataset["description"] = "Cadena de Markov tonal con 8 notas."
    return dataset


def tokenize_spanish(text):
    return re.findall(r"[a-záéíóúñü]+", text.lower())


def build_quijote_case():
    text = dedent("""
    En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo
    de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. Del poco dormir y del mucho leer
    se le secó el cerebro, de manera que vino a perder el juicio. Llenósele la fantasía de todo aquello que leía
    en los libros de caballerías, así de encantamientos como de pendencias, batallas, desafíos, heridas, requiebros,
    amores, tormentas y disparates imposibles.

    Y cuando estuvo del todo loco, le pareció convenible hacerse caballero andante y salir por el mundo con sus armas
    y caballo a buscar aventuras. Quiso llamarse don Quijote de la Mancha, y a su rocín quiso llamar Rocinante,
    nombre a su parecer alto, sonoro y significativo. Buscó también una dama de quien enamorarse, porque el caballero
    andante sin amores era árbol sin hojas y sin fruto. A esta dama vino a llamarla Dulcinea del Toboso.

    En esto, descubrieron treinta o cuarenta molinos de viento que hay en aquel campo, y así como don Quijote los vio,
    dijo a su escudero Sancho que la ventura iba guiando sus cosas mejor de lo que acertara a desear; porque allí se
    descubrían gigantes con quien pensaba hacer batalla. Respondió Sancho que aquellos que allí se parecían no eran
    gigantes, sino molinos de viento, y lo que en ellos parecían brazos eran las aspas.

    No obstante, don Quijote arremetió a todo galope con la lanza en ristre contra el primer molino. Le acompañaba
    Sancho Panza, fiel escudero, que procuraba advertirle del engaño. Después del golpe, quedaron en el relato
    Rocinante, Sancho, Dulcinea, los molinos y la obstinación de don Quijote como símbolos inseparables de la novela.
    """)

    stopwords = {
        "de", "la", "que", "y", "el", "en", "a", "los", "del", "se", "le", "lo", "por", "con", "como",
        "no", "ha", "un", "una", "su", "sus", "al", "era", "sin", "todo", "también", "allí", "aquellos",
        "aquello", "asi", "así", "hay", "muy", "o", "pero", "ya", "porque", "quien", "qué", "fue", "eran",
        "contra", "después", "las", "esto", "aquel", "aquella", "aquí", "allá", "este", "esta", "estos",
        "estas", "uno"
    }

    focus_tokens = [
        "quijote", "sancho", "rocinante", "dulcinea", "molinos", "viento", "caballero",
        "aventuras", "mancha", "gigantes", "batalla", "escudero", "novela", "juicio"
    ]

    tokens = [tok for tok in tokenize_spanish(text) if tok not in stopwords and len(tok) > 2]
    dataset = prepare_dataset_from_sequence(
        tokens,
        dataset_name="quijote_clasico",
        window_size=2,
        min_count=1,
        keep_tokens=focus_tokens,
    )
    dataset["description"] = "Fragmento clásico de Don Quijote con personajes y conceptos narrativos."
    return dataset


def describe_dataset(dataset, top_k=8):
    print("=" * 80)
    print(dataset["name"])
    print(dataset["description"])
    print("=" * 80)
    print("Vocabulario:", dataset["vocab"])
    print("Longitud de la secuencia filtrada:", len(dataset["sequence"]))
    print("Número de pares positivos únicos:", len(dataset["positive_pairs"]))
    print("Pares más frecuentes:")
    for (i, j), count in dataset["pair_counts"].most_common(top_k):
        print(f"  ({dataset['id_to_token'][i]}, {dataset['id_to_token'][j]}): {count}")


In [ ]:
datasets = {
    "musica_sintetica": build_music_case(),
    "quijote_clasico": build_quijote_case(),
}

for dataset in datasets.values():
    describe_dataset(dataset)


## 3. Planteamiento matemático del problema

Sea un vocabulario discreto $V=\{1,\dots,n\}$ y un embedding $E\in\mathbb{R}^{n\times d}$, donde cada fila $E_i$ representa un token.

El término de ajuste a datos sigue la lógica skip-gram con muestreo negativo:

$$
\mathcal{L}_{data}(E)
=-\sum_{(i,j)\in\mathcal{D}_+} w_{ij}\log\sigma(E_i^\top E_j)
-\sum_{(i,k)\in\mathcal{D}_-}\log\sigma(-E_i^\top E_k).
$$

Buscamos minimizar esa pérdida imponiendo dos restricciones geométricas:

$$
\|E_i\|_2^2 \leq c \quad \forall i,
\qquad E^\top E = I_d.
$$

La primera controla la compacidad de cada embedding y la segunda fuerza desacorrelación global entre dimensiones. El Lagrangiano asociado es

$$
\mathcal{L}(E,\lambda,\Lambda)
=\mathcal{L}_{data}(E)
+\sum_{i=1}^n \lambda_i(\|E_i\|_2^2-c)
+\langle \Lambda, E^\top E-I\rangle,
$$

con $\lambda_i\geq 0$ y $\Lambda$ simétrica. La comparación entre datasets es útil porque las condiciones KKT son las mismas, pero la geometría inducida por los datos cambia.


In [ ]:
def sample_negative_pairs(num_pairs, n_tokens):
    # Muestrea pares negativos uniformemente.
    centers = torch.randint(low=0, high=n_tokens, size=(num_pairs,))
    contexts = torch.randint(low=0, high=n_tokens, size=(num_pairs,))
    return torch.stack([centers, contexts], dim=1)


def data_loss(E, positive_pairs, positive_weights, num_negative=256):
    # Pérdida logística tipo skip-gram con muestreo negativo.
    i_pos = positive_pairs[:, 0]
    j_pos = positive_pairs[:, 1]
    logits_pos = (E[i_pos] * E[j_pos]).sum(dim=1)
    loss_pos = -(positive_weights * F.logsigmoid(logits_pos)).mean()

    neg_pairs = sample_negative_pairs(num_negative, E.shape[0])
    i_neg = neg_pairs[:, 0]
    j_neg = neg_pairs[:, 1]
    logits_neg = (E[i_neg] * E[j_neg]).sum(dim=1)
    loss_neg = -F.logsigmoid(-logits_neg).mean()

    return loss_pos + loss_neg


def orthogonality_penalty(E):
    d = E.shape[1]
    gram = E.T @ E
    I = torch.eye(d, dtype=E.dtype, device=E.device)
    return torch.sum((gram - I) ** 2)


def norm_violation_penalty(E, c):
    squared_norms = torch.sum(E ** 2, dim=1)
    violations = torch.clamp(squared_norms - c, min=0.0)
    return torch.sum(violations ** 2)


def log_barrier_norm(E, c, eps=1e-8):
    squared_norms = torch.sum(E ** 2, dim=1)
    slack = c - squared_norms
    if torch.any(slack <= 0):
        return torch.tensor(float("inf"), dtype=E.dtype, device=E.device)
    return -torch.sum(torch.log(slack + eps))


def summarize_constraints(E, c=1.0):
    norms_sq = torch.sum(E ** 2, dim=1)
    gram = E.T @ E
    return {
        "max_norm_sq": norms_sq.max().item(),
        "min_slack": (c - norms_sq).min().item(),
        "orth_error": torch.linalg.norm(gram - torch.eye(E.shape[1])).item(),
        "gram": gram.detach().numpy(),
    }


## 4. Métodos numéricos

Usaremos dos aproximaciones del curso:

1. **Penalización cuadrática**
$$
\min_E \; \mathcal{L}_{data}(E)
+ \alpha \|E^\top E - I\|_F^2
+ \beta \sum_i \max(0, \|E_i\|^2-c)^2.
$$

2. **Barrera logarítmica**
$$
\min_E \; \mathcal{L}_{data}(E)
+ \alpha \|E^\top E - I\|_F^2
- \mu \sum_i \log(c-\|E_i\|^2).
$$

La diferencia central es que la penalización tolera pequeñas violaciones durante el entrenamiento, mientras que la barrera empuja a permanecer en el interior del conjunto factible.


In [ ]:
def train_quadratic_penalty(
    dataset,
    dim=2,
    c=1.0,
    alpha=1.0,
    beta=10.0,
    lr=0.03,
    epochs=900,
    print_every=300,
):
    E = torch.nn.Parameter(0.1 * torch.randn(dataset["n_tokens"], dim))
    optimizer = torch.optim.Adam([E], lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        loss_data = data_loss(E, dataset["positive_pairs"], dataset["positive_weights"])
        loss_orth = orthogonality_penalty(E)
        loss_norm = norm_violation_penalty(E, c)
        loss = loss_data + alpha * loss_orth + beta * loss_norm
        loss.backward()
        optimizer.step()

        if epoch % print_every == 0 or epoch == 1:
            with torch.no_grad():
                summary = summarize_constraints(E, c=c)
                history.append({
                    "epoch": epoch,
                    "loss": loss.item(),
                    "data_loss": loss_data.item(),
                    "orth_error": summary["orth_error"],
                    "max_norm_sq": summary["max_norm_sq"],
                })
                print(
                    f"[{dataset['name']}] penalty epoch={epoch:4d} | "
                    f"loss={loss.item():.4f} | data={loss_data.item():.4f} | "
                    f"orth_error={summary['orth_error']:.4f} | max_norm_sq={summary['max_norm_sq']:.4f}"
                )

    return E.detach(), history


def train_log_barrier(
    dataset,
    dim=2,
    c=1.0,
    alpha=1.0,
    mu=0.03,
    lr=0.01,
    epochs=900,
    print_every=300,
):
    E = torch.nn.Parameter(0.15 * torch.randn(dataset["n_tokens"], dim))
    optimizer = torch.optim.Adam([E], lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        loss_data = data_loss(E, dataset["positive_pairs"], dataset["positive_weights"])
        loss_orth = orthogonality_penalty(E)
        loss_barrier = log_barrier_norm(E, c)
        loss = loss_data + alpha * loss_orth + mu * loss_barrier
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            norms = torch.linalg.norm(E, dim=1, keepdim=True)
            max_allowed = math.sqrt(c) * 0.999
            scale = torch.clamp(max_allowed / (norms + 1e-12), max=1.0)
            E.mul_(scale)

        if epoch % print_every == 0 or epoch == 1:
            with torch.no_grad():
                summary = summarize_constraints(E, c=c)
                history.append({
                    "epoch": epoch,
                    "loss": loss.item(),
                    "data_loss": loss_data.item(),
                    "barrier": loss_barrier.item(),
                    "orth_error": summary["orth_error"],
                    "max_norm_sq": summary["max_norm_sq"],
                })
                print(
                    f"[{dataset['name']}] barrier epoch={epoch:4d} | "
                    f"loss={loss.item():.4f} | data={loss_data.item():.4f} | "
                    f"barrier={loss_barrier.item():.4f} | orth_error={summary['orth_error']:.4f} | "
                    f"max_norm_sq={summary['max_norm_sq']:.4f}"
                )

    return E.detach(), history


def run_experiment(dataset, dim=2, c=1.0):
    E_penalty, hist_penalty = train_quadratic_penalty(dataset, dim=dim, c=c)
    E_barrier, hist_barrier = train_log_barrier(dataset, dim=dim, c=c)
    return {
        "dataset": dataset,
        "penalty": {
            "E": E_penalty,
            "history": hist_penalty,
            "summary": summarize_constraints(E_penalty, c=c),
        },
        "barrier": {
            "E": E_barrier,
            "history": hist_barrier,
            "summary": summarize_constraints(E_barrier, c=c),
        },
    }


In [ ]:
results = {name: run_experiment(dataset) for name, dataset in datasets.items()}


## 5. Visualización de embeddings

Como usamos dimensión $d=2$, podemos comparar visualmente los cuatro embeddings aprendidos: dos datasets y dos métodos. El círculo representa la frontera asociada a la restricción $\|E_i\|^2 \leq c$.


In [ ]:
def plot_embeddings(ax, E, dataset, title, c=1.0):
    E_np = E.numpy()
    ax.scatter(E_np[:, 0], E_np[:, 1], s=45)

    for idx, label in dataset["id_to_token"].items():
        ax.annotate(label, (E_np[idx, 0], E_np[idx, 1]), xytext=(5, 5), textcoords="offset points", fontsize=9)

    circle = plt.Circle((0, 0), math.sqrt(c), fill=False, linestyle="--")
    ax.add_patch(circle)
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("Dimensión 1")
    ax.set_ylabel("Dimensión 2")
    margin = 1.2 * math.sqrt(c)
    ax.set_xlim(-margin, margin)
    ax.set_ylim(-margin, margin)


fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for row, (dataset_name, experiment) in enumerate(results.items()):
    plot_embeddings(
        axes[row, 0],
        experiment["penalty"]["E"],
        experiment["dataset"],
        f"{dataset_name} | penalización cuadrática",
    )
    plot_embeddings(
        axes[row, 1],
        experiment["barrier"]["E"],
        experiment["dataset"],
        f"{dataset_name} | barrera logarítmica",
    )

plt.tight_layout()
plt.show()


In [ ]:
def plot_history_grid(results, metric, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for ax, (dataset_name, experiment) in zip(axes, results.items()):
        for method_name, color in [("penalty", "tab:blue"), ("barrier", "tab:orange")]:
            history = experiment[method_name]["history"]
            epochs = [h["epoch"] for h in history]
            values = [h[metric] for h in history]
            ax.plot(epochs, values, marker="o", label=method_name, color=color)
        ax.set_title(dataset_name)
        ax.set_xlabel("Época")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()


plot_history_grid(results, metric="data_loss", ylabel="Pérdida de datos")
plot_history_grid(results, metric="orth_error", ylabel="Error de ortogonalidad")
plot_history_grid(results, metric="max_norm_sq", ylabel="Máxima norma cuadrada")


## 6. Comparación cuantitativa

Aquí resumimos la factibilidad y el ajuste final de cada combinación `dataset + método`. Lo importante es comparar dos ejes:

- Qué tan bien ajusta las co-ocurrencias observadas.
- Qué tan cerca queda de satisfacer las restricciones del problema original.


In [ ]:
def final_data_loss(E, dataset):
    with torch.no_grad():
        return data_loss(E, dataset["positive_pairs"], dataset["positive_weights"]).item()


print(f"{'dataset':<20} {'metodo':<12} {'data_loss':>10} {'orth_error':>12} {'max_norm_sq':>12} {'min_slack':>12}")
print("-" * 82)
for dataset_name, experiment in results.items():
    dataset = experiment["dataset"]
    for method_name in ["penalty", "barrier"]:
        summary = experiment[method_name]["summary"]
        data_term = final_data_loss(experiment[method_name]["E"], dataset)
        print(
            f"{dataset_name:<20} {method_name:<12} "
            f"{data_term:10.4f} {summary['orth_error']:12.4f} {summary['max_norm_sq']:12.4f} {summary['min_slack']:12.4f}"
        )


In [ ]:
def nearest_neighbors(E, dataset, top_k=3):
    with torch.no_grad():
        normalized = E / torch.linalg.norm(E, dim=1, keepdim=True).clamp_min(1e-12)
        similarity = normalized @ normalized.T

    neighbors = {}
    for idx, token in dataset["id_to_token"].items():
        sims = similarity[idx].clone()
        sims[idx] = -1e9
        top_indices = torch.topk(sims, k=min(top_k, len(sims) - 1)).indices.tolist()
        neighbors[token] = [dataset["id_to_token"][j] for j in top_indices]
    return neighbors


for dataset_name, experiment in results.items():
    print("=" * 80)
    print(f"Vecinos más cercanos en {dataset_name} (método de barrera)")
    print("=" * 80)
    for token, nbrs in nearest_neighbors(experiment["barrier"]["E"], experiment["dataset"]).items():
        print(f"{token:>12s} -> {', '.join(nbrs)}")


## 7. Interpretación comparativa

La comparación ahora es más defendible para un proyecto final:

- El caso `musica_sintetica` sigue siendo útil como control porque sabemos cómo se generó la secuencia y podemos verificar si el embedding recupera proximidades tonales plausibles.
- El caso `quijote_clasico` es más interesante porque introduce co-ocurrencias con significado narrativo: por ejemplo, `quijote`, `rocinante`, `sancho`, `molinos` y `viento` deberían tender a agruparse si el modelo captura bien el contexto.
- Si ambos métodos producen pérdidas de datos parecidas, pero la barrera deja mayor holgura interior, eso sugiere una mejor aproximación a la factibilidad primal. Si la penalización ajusta más rápido pero roza más el borde, eso también es una observación valiosa desde optimización restringida.
- El punto importante no es solo “qué método gana”, sino cómo cambia la geometría del embedding cuando los datos dejan de ser puramente sintéticos y pasan a representar un caso clásico interpretable.


## 8. Extensiones naturales

Tres extensiones razonables para seguir mejorando el notebook son:

1. Sustituir el fragmento incrustado de *Don Quijote* por un corpus completo o por varios capítulos para obtener estadísticas más estables.
2. Incorporar una métrica externa de interpretación, por ejemplo revisar vecinos esperados para ciertos tokens ancla.
3. Añadir una restricción extra de separación entre grupos semánticos si quieres conectar el experimento con clusters predefinidos.

Con esta versión ya tienes un contraste claro entre un experimento controlado y un caso clásico, manteniendo la misma formulación matemática y la misma discusión en términos de factibilidad, penalización, barrera y KKT.
